In [ ]:
# Cell 1: Imports & Setup
import sys
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
from pathlib import Path
import torch
from transformers import Seq2SeqTrainer

# Ensure the package is importable
notebook_dir = Path.cwd()
whisper_dir = notebook_dir.parent
sys.path.append(str(whisper_dir))

from whisper_asr import (
    load_idiom_data,
    OnTheFlyDataset,
    collate_fn,
    load_model_and_processor,
    compute_metrics,
    get_training_args,
)
from whisper_asr.utils import get_best_gpu

# Disable tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

#DEVICE = torch.device(f"cuda:{get_best_gpu()}" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda")
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
# Cell 2: Load Data (now returns plain lists)

print("Loading all Romansh idiom datasets...")
train_samples, val_samples = load_idiom_data()
print(f"Train samples: {len(train_samples)}, Validation samples: {len(val_samples)}")

Loading all Romansh idiom datasets...


Train samples: 29565, Validation samples: 3288


In [4]:
# Cell 3: Load Model & Processor (with Italian language fixed)
MODEL_NAME = "openai/whisper-medium"
OUTPUT_DIR = "../models/whisper-medium-rm-all-it"

model, feature_extractor, tokenizer, processor = load_model_and_processor(
    MODEL_NAME, DEVICE, language="it"
)
print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

Loading weights: 100%|██████████| 947/947 [00:00<00:00, 7392.52it/s]


Model loaded: 763.9M params


In [5]:
# Cell 4: Create On‑the‑Fly Datasets (pass the lists directly)
train_dataset = OnTheFlyDataset(
    train_samples,          # list of dicts
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    language="it",
    task="transcribe",
)
eval_dataset = OnTheFlyDataset(
    val_samples,
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    language="it",
    task="transcribe",
)
print(f"Train dataset size: {len(train_dataset)}, Eval dataset size: {len(eval_dataset)}")

Train dataset size: 29565, Eval dataset size: 3288


In [6]:
# Cell 5: Training Arguments
training_args = get_training_args(OUTPUT_DIR)
print("Training arguments:")
print(training_args)

Training arguments:
Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1000,
eval_strategy=I

In [7]:
# Cell 6: Define a wrapper for compute_metrics that already knows the tokenizer.
from functools import partial
compute_metrics_fn = partial(compute_metrics, tokenizer=tokenizer)

In [8]:
# Cell 7: Initialize Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics_fn,
)
print("Trainer initialized.")

Trainer initialized.


In [ ]:
# Cell 9: Start Training
print("Starting training...")
trainer.train()

Starting training...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

: 

In [ ]:
# Cell 10: Save Final Model
print(f"Saving model to {OUTPUT_DIR}")
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Done.")